In [1]:
import mysql.connector
import pandas
import sqlalchemy
import pymysql

print("All MySQL libraries installed successfully!")

All MySQL libraries installed successfully!


In [2]:
import pandas as pd
import mysql.connector
from sqlalchemy import create_engine
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [6]:
import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password="1234"
)

print("MySQL Connected Successfully!")

MySQL Connected Successfully!


In [ ]:
#  database creation

cursor = connection.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS ecommerce_db")

cursor.execute("USE ecommerce_db")

print("Database created successfully!")

cursor.close()
connection.close()

Database created successfully!


In [9]:
# Cell 4 - Create MySQL engine

from sqlalchemy import create_engine, URL

database_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password=1234,
    host="localhost",
    port=3306,
    database="ecommerce_db"
)

engine = create_engine(database_url)

with engine.connect() as conn:
    print("MySQL database is ready!")

MySQL database is ready!


In [11]:
# Import all cleaned datasets into MySQL

# Cell 5 - Import cleaned CSV files into MySQL

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "processed"

files = {
    "orders_clean": "orders_clean.csv",
    "customers_clean": "customers_clean.csv",
    "order_items_clean": "order_items_clean.csv",
    "payments_clean": "payments_clean.csv",
    "products_clean": "products_clean.csv",
    "reviews_clean": "reviews_clean.csv",
    "sellers_clean": "sellers_clean.csv"
}

for table, filename in files.items():

    file_path = PROCESSED_DIR / filename

    df = pd.read_csv(file_path)

    df.to_sql(
        name=table,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=1000
    )

    print(table, "imported successfully!")

print("ALL DATASETS IMPORTED INTO MYSQL!")

# This may take a little time because the Olist dataset contains many rows.

orders_clean imported successfully!
customers_clean imported successfully!
order_items_clean imported successfully!
payments_clean imported successfully!
products_clean imported successfully!
reviews_clean imported successfully!
sellers_clean imported successfully!
ALL DATASETS IMPORTED INTO MYSQL!


In [12]:
# Step 3 — Verify imported tables

# Cell 6 - Verify MySQL tables

query = "SHOW TABLES"

tables = pd.read_sql(query, engine)

display(tables)

,Tables_in_ecommerce_db
0,customers_clean
1,order_items_clean
2,orders_clean
3,payments_clean
4,products_clean
5,reviews_clean
6,sellers_clean


In [13]:
#  Basic SQL business queries
# Use this Python helper to execute SQL queries from your notebook:

def run_sql(query):
    result = pd.read_sql_query(query, engine)
    display(result)

In [14]:
# Query 1 - Total Orders

query = """
SELECT
    COUNT(DISTINCT order_id) AS total_orders
FROM orders_clean;
"""

run_sql(query)

,total_orders
0,99441


In [15]:
# Query 2 - Total Unique Customers

query = """
SELECT
    COUNT(DISTINCT customer_unique_id) AS total_customers
FROM customers_clean;
"""

run_sql(query)

,total_customers
0,96096


In [16]:
# Query 3 - Total Product Revenue

query = """
SELECT
    ROUND(SUM(price), 2) AS total_product_revenue
FROM order_items_clean;
"""

run_sql(query)

,total_product_revenue
0,13591643.7


In [18]:
# Query 4 - Total Payment Value

query = """
SELECT
    ROUND(SUM(payment_value), 2) AS total_payment_value
FROM payments_clean;
"""

run_sql(query)
# Payment value and product revenue are different metrics. Do not add them together.

,total_payment_value
0,16008872.12


In [19]:
# Query 5 - Order Status Analysis

query = """
SELECT
    order_status,
    COUNT(*) AS total_orders
FROM orders_clean
GROUP BY order_status
ORDER BY total_orders DESC;
"""

run_sql(query)

,order_status,total_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [ ]:
# Run a simpler monthly revenue query
query = """
SELECT
    YEAR(o.order_purchase_timestamp) AS year,
    MONTH(o.order_purchase_timestamp) AS month,
    SUM(i.price) AS monthly_revenue
FROM orders_clean AS o
JOIN order_items_clean AS i
    ON o.order_id = i.order_id
GROUP BY
    YEAR(o.order_purchase_timestamp),
    MONTH(o.order_purchase_timestamp)
ORDER BY year, month
"""

run_sql(query)

,year,month,monthly_revenue
0,2016,9,267.36
1,2016,10,49507.66
2,2016,12,10.90
3,2017,1,120312.87
4,2017,2,247303.02
5,2017,3,374344.30
6,2017,4,359927.23
7,2017,5,506071.14
8,2017,6,433038.60
9,2017,7,498031.48


In [26]:
# Query 6 - Monthly Revenue Analysis

query = """
SELECT
    YEAR(o.order_purchase_timestamp) AS year,
    MONTH(o.order_purchase_timestamp) AS month,
    ROUND(SUM(i.price), 2) AS monthly_revenue

FROM orders_clean AS o

JOIN order_items_clean AS i
    ON o.order_id = i.order_id

WHERE o.order_status = 'delivered'

GROUP BY
    YEAR(o.order_purchase_timestamp),
    MONTH(o.order_purchase_timestamp)

ORDER BY year, month;
"""

run_sql(query)

,year,month,monthly_revenue
0,2016,9,134.97
1,2016,10,40325.11
2,2016,12,10.90
3,2017,1,111798.36
4,2017,2,234223.40
5,2017,3,359198.85
6,2017,4,340669.68
7,2017,5,489338.25
8,2017,6,421923.37
9,2017,7,481604.52


In [27]:
# Query 7 - Top 10 Product Categories

query = """
SELECT
    p.product_category_name,
    ROUND(SUM(i.price), 2) AS total_revenue,
    COUNT(DISTINCT i.order_id) AS total_orders
FROM order_items_clean AS i

JOIN products_clean AS p
    ON i.product_id = p.product_id

GROUP BY
    p.product_category_name

ORDER BY
    total_revenue DESC

LIMIT 10;
"""

run_sql(query)

,product_category_name,total_revenue,total_orders
0,beleza_saude,1258681.34,8836
1,relogios_presentes,1205005.68,5624
2,cama_mesa_banho,1036988.68,9417
3,esporte_lazer,988048.97,7720
4,informatica_acessorios,911954.32,6689
5,moveis_decoracao,729762.49,6449
6,cool_stuff,635290.85,3632
7,utilidades_domesticas,632248.66,5884
8,automotivo,592720.11,3897
9,ferramentas_jardim,485256.46,3518


In [28]:
# Query 8 - Payment Method Analysis

query = """
SELECT
    payment_type,
    COUNT(*) AS payment_count,
    ROUND(SUM(payment_value), 2) AS total_payment
FROM payments_clean
GROUP BY payment_type
ORDER BY total_payment DESC;
"""

run_sql(query)

,payment_type,payment_count,total_payment
0,credit_card,76795,12542084.19
1,boleto,19784,2869361.27
2,voucher,5775,379436.87
3,debit_card,1529,217989.79
4,not_defined,3,0.00


In [29]:
# Query 9 - Review Analysis

query = """
SELECT
    review_score,
    COUNT(*) AS total_reviews,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM reviews_clean), 2
    ) AS review_percentage
FROM reviews_clean
GROUP BY review_score
ORDER BY review_score;
"""

run_sql(query)

,review_score,total_reviews,review_percentage
0,1,11424,11.51
1,2,3151,3.18
2,3,8179,8.24
3,4,19142,19.29
4,5,57328,57.78


In [30]:
# Query 10 - Average Delivery Time

query = """
SELECT
    ROUND(
        AVG(DATEDIFF(
            order_delivered_customer_date,
            order_purchase_timestamp
        )), 2
    ) AS average_delivery_days
FROM orders_clean
WHERE order_status = 'delivered'
AND order_delivered_customer_date IS NOT NULL;
"""

run_sql(query)

,average_delivery_days
0,12.5


In [31]:
#  Advanced SQL Analysis
# # ============================================================
# INTERVIEW ANSWER — CTE
# ============================================================
#
# A CTE stands for Common Table Expression.
#
# A CTE is a temporary result set defined using
# the WITH keyword.
#
# It helps simplify complex SQL queries.
#
# It improves SQL query readability.
#
# It makes complex queries easier to understand
# and maintain.
#
# ============================================================Now we'll move to CTEs and Window Functions. These are important skills for Data Analyst interviews.

In [32]:
# Query 11 - CTE: Top 10 Product Categories

query = """
WITH category_sales AS (
    SELECT
        p.product_category_name,
        SUM(i.price) AS total_revenue,
        COUNT(DISTINCT i.order_id) AS total_orders
    FROM order_items_clean i
    JOIN products_clean p
        ON i.product_id = p.product_id
    GROUP BY p.product_category_name
)

SELECT
    product_category_name,
    ROUND(total_revenue, 2) AS total_revenue,
    total_orders
FROM category_sales
ORDER BY total_revenue DESC
LIMIT 10;
"""

run_sql(query)

,product_category_name,total_revenue,total_orders
0,beleza_saude,1258681.34,8836
1,relogios_presentes,1205005.68,5624
2,cama_mesa_banho,1036988.68,9417
3,esporte_lazer,988048.97,7720
4,informatica_acessorios,911954.32,6689
5,moveis_decoracao,729762.49,6449
6,cool_stuff,635290.85,3632
7,utilidades_domesticas,632248.66,5884
8,automotivo,592720.11,3897
9,ferramentas_jardim,485256.46,3518


In [33]:
# Query 12 — Window Function: Rank Product Categories

# Query 12 - Rank Product Categories

query = """
WITH category_sales AS (
    SELECT
        p.product_category_name,
        SUM(i.price) AS total_revenue
    FROM order_items_clean i
    JOIN products_clean p
        ON i.product_id = p.product_id
    GROUP BY p.product_category_name
)

SELECT
    product_category_name,
    ROUND(total_revenue, 2) AS total_revenue,
    DENSE_RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS category_rank
FROM category_sales
ORDER BY category_rank;
"""

run_sql(query)

,product_category_name,total_revenue,category_rank
0,beleza_saude,1258681.34,1
1,relogios_presentes,1205005.68,2
2,cama_mesa_banho,1036988.68,3
3,esporte_lazer,988048.97,4
4,informatica_acessorios,911954.32,5
...,...,...,...
69,flores,1110.04,70
70,casa_conforto_2,760.27,71
71,cds_dvds_musicais,730.00,72
72,fashion_roupa_infanto_juvenil,569.85,73


In [34]:
#   Query 13 — Window Function: Top 10 Customers

# Query 13 - Rank Top Customers

query = """
WITH customer_sales AS (
    SELECT
        c.customer_unique_id,
        SUM(i.price) AS total_spending,
        COUNT(DISTINCT o.order_id) AS total_orders
    FROM customers_clean c
    JOIN orders_clean o
        ON c.customer_id = o.customer_id
    JOIN order_items_clean i
        ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)

SELECT
    customer_unique_id,
    ROUND(total_spending, 2) AS total_spending,
    total_orders,
    DENSE_RANK() OVER (
        ORDER BY total_spending DESC
    ) AS customer_rank
FROM customer_sales
ORDER BY customer_rank
LIMIT 10;
"""

run_sql(query)

,customer_unique_id,total_spending,total_orders,customer_rank
0,0a0a92112bd4c708ca5fde585afaa872,13440.0,1,1
1,da122df9eeddfedc1dc1f5349a1a690c,7388.0,2,2
2,763c8b1c9c68a0229c42c9fc6f662b93,7160.0,1,3
3,dc4802a71eae9be1dd28f5d788ceb526,6735.0,1,4
4,459bef486812aa25204be022145caa62,6729.0,1,5
5,ff4159b92c40ebe40454e3e6a7c35ed6,6499.0,1,6
6,4007669dec559734d6f53e029e360987,5934.6,1,7
7,eebb5dda148d3893cdaf5b5ca3040ccb,4690.0,1,8
8,48e1ac109decbb87765a3eade6854098,4590.0,1,9
9,a229eba70ec1c2abef51f04987deb7a5,4400.0,1,10


In [35]:
# Query 14 — LAG(): Monthly Revenue Growth

# Query 14 - Month-over-Month Revenue Growth

query = """
WITH monthly_sales AS (
    SELECT
        YEAR(o.order_purchase_timestamp) AS year,
        MONTH(o.order_purchase_timestamp) AS month,
        SUM(i.price) AS revenue
    FROM orders_clean o
    JOIN order_items_clean i
        ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY
        YEAR(o.order_purchase_timestamp),
        MONTH(o.order_purchase_timestamp)
),

previous_sales AS (
    SELECT
        year,
        month,
        revenue,
        LAG(revenue) OVER (
            ORDER BY year, month
        ) AS previous_revenue
    FROM monthly_sales
)

SELECT
    year,
    month,
    ROUND(revenue, 2) AS current_revenue,
    ROUND(previous_revenue, 2) AS previous_revenue,
    ROUND(
        (revenue - previous_revenue)
        * 100 / NULLIF(previous_revenue, 0),
        2
    ) AS growth_percentage
FROM previous_sales
ORDER BY year, month;
"""

run_sql(query)

,year,month,current_revenue,previous_revenue,growth_percentage
0,2016,9,134.97,NaN,NaN
1,2016,10,40325.11,134.97,29777.09
2,2016,12,10.90,40325.11,-99.97
3,2017,1,111798.36,10.90,1025573.03
4,2017,2,234223.40,111798.36,109.51
5,2017,3,359198.85,234223.40,53.36
6,2017,4,340669.68,359198.85,-5.16
7,2017,5,489338.25,340669.68,43.64
8,2017,6,421923.37,489338.25,-13.78
9,2017,7,481604.52,421923.37,14.15


In [36]:
# Query 15 — Running Total Revenue

# Query 15 - Cumulative Revenue

query = """
WITH monthly_sales AS (
    SELECT
        YEAR(o.order_purchase_timestamp) AS year,
        MONTH(o.order_purchase_timestamp) AS month,
        SUM(i.price) AS revenue
    FROM orders_clean o
    JOIN order_items_clean i
        ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY
        YEAR(o.order_purchase_timestamp),
        MONTH(o.order_purchase_timestamp)
)

SELECT
    year,
    month,
    ROUND(revenue, 2) AS monthly_revenue,
    ROUND(
        SUM(revenue) OVER (
            ORDER BY year, month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2
    ) AS cumulative_revenue
FROM monthly_sales
ORDER BY year, month;
"""

run_sql(query)

,year,month,monthly_revenue,cumulative_revenue
0,2016,9,134.97,134.97
1,2016,10,40325.11,40460.08
2,2016,12,10.90,40470.98
3,2017,1,111798.36,152269.34
4,2017,2,234223.40,386492.74
5,2017,3,359198.85,745691.59
6,2017,4,340669.68,1086361.27
7,2017,5,489338.25,1575699.52
8,2017,6,421923.37,1997622.89
9,2017,7,481604.52,2479227.41


In [37]:
#Advanced SQL interview revision
#Function	Purpose
#CTE	Create temporary named query results
#RANK()	Rank records with gaps after ties
#DENSE_RANK()	Rank records without gaps
#LAG()	Access previous row's value
#LEAD()	Access next row's value
#SUM() OVER()	Calculate running totals

In [38]:
#STEP 07 — Advanced SQL Business Analysis
#Continue in 07_sql_business_analysis.ipynb.
#Query 16 — State-wise Revenue Analysis
#Business question: Which Brazilian states generate the highest product revenue?

In [39]:
# Query 16 - State-wise Revenue

query = """
SELECT
    c.customer_state,
    ROUND(SUM(i.price), 2) AS total_revenue,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM customers_clean c
JOIN orders_clean o
    ON c.customer_id = o.customer_id
JOIN order_items_clean i
    ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY total_revenue DESC
LIMIT 10;
"""

run_sql(query)

,customer_state,total_revenue,total_orders
0,SP,5067633.16,40501
1,RJ,1759651.13,12350
2,MG,1552481.83,11354
3,RS,728897.47,5345
4,PR,666063.51,4923
5,SC,507012.13,3546
6,BA,493584.14,3256
7,DF,296498.41,2080
8,GO,282836.70,1957
9,ES,268643.45,1995


In [40]:
# Query 17 — Seller Performance Analysis
# Business question: Which sellers generate the highest revenue?

# Query 17 - Seller Performance

query = """
SELECT
    seller_id,
    ROUND(SUM(price), 2) AS total_revenue,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(AVG(price), 2) AS average_item_price
FROM order_items_clean
GROUP BY seller_id
ORDER BY total_revenue DESC
LIMIT 10;
"""

run_sql(query)

,seller_id,total_revenue,total_orders,average_item_price
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1132,198.51
1,53243585a1d6dc2643021fd1853d8905,222776.05,358,543.36
2,4a3ca9315b744ce9f8e9374361493884,200472.92,1806,100.89
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,585,331.13
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,982,137.77
5,7e93a43ef30c4f03f38b393420bc753a,176431.87,336,518.92
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,1314,103.31
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53,1160,121.05
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,915,97.32
9,955fee9216a65b617aa5c0531780ce60,135171.70,1287,90.17


In [41]:
#  Query 18 — Late Delivery Analysis
#Business question: How many orders were delivered late?

In [42]:
# Query 18 - Late Delivery Analysis

query = """
SELECT
    COUNT(*) AS total_delivered_orders,

    SUM(
        CASE
            WHEN order_delivered_customer_date >
                 order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN order_delivered_customer_date >
                     order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(*), 2
    ) AS late_delivery_percentage

FROM orders_clean
WHERE order_status = 'delivered'
AND order_delivered_customer_date IS NOT NULL
AND order_estimated_delivery_date IS NOT NULL;
"""

run_sql(query)

,total_delivered_orders,late_orders,late_delivery_percentage
0,96470,7826.0,8.11


In [43]:
#  Query 19 — Review Score by Delivery Performance
#Business question: Do late deliveries have lower customer review scores?

# Query 19 - Delivery Performance vs Review Score

query = """
SELECT
    CASE
        WHEN o.order_delivered_customer_date >
             o.order_estimated_delivery_date
        THEN 'Late Delivery'
        ELSE 'On-Time Delivery'
    END AS delivery_status,

    ROUND(AVG(r.review_score), 2) AS average_review_score,
    COUNT(*) AS total_reviews

FROM orders_clean o

JOIN reviews_clean r
    ON o.order_id = r.order_id

WHERE o.order_status = 'delivered'
AND o.order_delivered_customer_date IS NOT NULL
AND o.order_estimated_delivery_date IS NOT NULL

GROUP BY delivery_status;
"""

run_sql(query)

,delivery_status,average_review_score,total_reviews
0,On-Time Delivery,4.29,88653
1,Late Delivery,2.57,7700


In [44]:
# Query 20 — Payment Installment Analysis
#Business question: Which installment plans are commonly used by customers?

# Query 20 - Payment Installment Analysis

query = """
SELECT
    payment_installments,
    COUNT(*) AS payment_count,
    ROUND(AVG(payment_value), 2) AS average_payment,
    ROUND(SUM(payment_value), 2) AS total_payment
FROM payments_clean
GROUP BY payment_installments
ORDER BY payment_installments;
"""

run_sql(query)

,payment_installments,payment_count,average_payment,total_payment
0,0,2,94.32,188.63
1,1,52546,112.42,5907233.36
2,2,12413,127.23,1579283.03
3,3,10461,142.54,1491103.80
4,4,7098,163.98,1163907.61
5,5,5239,183.47,961174.30
6,6,3920,209.85,822611.81
7,7,1626,187.67,305157.39
8,8,4268,307.74,1313423.34
9,9,644,203.44,131015.92


In [45]:
# STEP 07 — Final SQL Export
#Once Queries 1–20 work, save important SQL results as CSV files for Excel and Power BI.
#Cell — Export SQL analysis results

In [46]:
# Export SQL results for Power BI and Excel

from pathlib import Path

REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(exist_ok=True)

export_queries = {
    "monthly_revenue": """
        SELECT
            YEAR(o.order_purchase_timestamp) AS year,
            MONTH(o.order_purchase_timestamp) AS month,
            ROUND(SUM(i.price), 2) AS revenue
        FROM orders_clean o
        JOIN order_items_clean i
            ON o.order_id = i.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY year, month
        ORDER BY year, month
    """,

    "state_revenue": """
        SELECT
            c.customer_state,
            ROUND(SUM(i.price), 2) AS revenue
        FROM customers_clean c
        JOIN orders_clean o
            ON c.customer_id = o.customer_id
        JOIN order_items_clean i
            ON o.order_id = i.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY c.customer_state
        ORDER BY revenue DESC
    """,

    "payment_analysis": """
        SELECT
            payment_type,
            ROUND(SUM(payment_value), 2) AS total_payment
        FROM payments_clean
        GROUP BY payment_type
        ORDER BY total_payment DESC
    """
}

for filename, sql_query in export_queries.items():

    result = pd.read_sql_query(sql_query, engine)

    result.to_csv(
        REPORT_DIR / f"{filename}.csv",
        index=False
    )

    print(filename, "exported successfully!")

print("SQL EXPORT COMPLETED!")

monthly_revenue exported successfully!
state_revenue exported successfully!
payment_analysis exported successfully!
SQL EXPORT COMPLETED!


In [2]:
# Verify exported CSV files

from pathlib import Path

# Define reports folder
PROJECT_ROOT = Path.cwd().parent
REPORT_DIR = PROJECT_ROOT / "reports"

# Check folder
print("Reports folder:", REPORT_DIR)
print("Folder exists:", REPORT_DIR.exists())

# List exported files
for file in REPORT_DIR.glob("*.csv"):
    print("Saved successfully:", file.name)

Reports folder: c:\Users\bapun\OneDrive\Desktop\E-Commerce-Data-Analytics\reports
Folder exists: True
Saved successfully: monthly_revenue.csv
Saved successfully: payment_analysis.csv
Saved successfully: state_revenue.csv
